In [0]:
import torch
import torch.nn as nn
import math

In [0]:
class StintLSTM(nn.Module):
    """
    LSTM-based lap time prediction model.
    Uses the SAME dataloader as StintTransformer (dataloader.py).
    
    Key differences from Transformer:
    - Uses LSTM instead of self-attention
    - Processes sequences recurrently (left-to-right)
    - Uses pack_padded_sequence for efficient padding handling
    """
    def __init__(self, n_drivers, n_teams, n_tyres, n_modes=3,
                 driver_emb_dim=8, team_emb_dim=8, tyre_emb_dim=4, mode_emb_dim=2,
                 n_cont_features=10, hidden_dim=64, num_layers=2, dropout=0.3):
        super().__init__()
        
        # Embeddings (same as Transformer)
        self.driver_emb = nn.Embedding(n_drivers, driver_emb_dim)
        self.team_emb = nn.Embedding(n_teams, team_emb_dim)
        self.tyre_emb = nn.Embedding(n_tyres, tyre_emb_dim)
        self.mode_emb = nn.Embedding(n_modes, mode_emb_dim)
        
        # Input dimension
        input_dim = n_cont_features + driver_emb_dim + team_emb_dim + tyre_emb_dim + mode_emb_dim
        
        # LSTM layers
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=False  # Set to True for BiLSTM
        )
        
        # Output layer
        self.output_layer = nn.Linear(hidden_dim, 1)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, cont_feats, driver_idx, team_idx, tyre_idx, mode_idx, mask=None):
        """
        Same input signature as StintTransformer - no changes needed to dataloader!
        
        Args:
            cont_feats: (batch_size, seq_len, n_cont_features)
            driver_idx: (batch_size,)
            team_idx: (batch_size,)
            tyre_idx: (batch_size, seq_len)
            mode_idx: (batch_size, seq_len)
            mask: (batch_size, seq_len) - True for padded positions
        
        Returns:
            out: (batch_size, seq_len) - predicted lap times
        """
        batch_size, seq_len, _ = cont_feats.size()
        
        # Create embeddings (same as Transformer)
        driver_emb = self.driver_emb(driver_idx).unsqueeze(1).repeat(1, seq_len, 1)
        team_emb = self.team_emb(team_idx).unsqueeze(1).repeat(1, seq_len, 1)
        tyre_emb = self.tyre_emb(tyre_idx)
        mode_emb = self.mode_emb(mode_idx)
        
        # Concatenate all features
        x = torch.cat([cont_feats, driver_emb, team_emb, tyre_emb, mode_emb], dim=-1)
        
        # Pack sequences for efficient LSTM processing (optional but faster)
        if mask is not None:
            # Calculate sequence lengths from mask
            seq_lengths = (~mask).sum(dim=1).cpu()  # Count non-padded positions
            
            # Pack padded sequences
            x_packed = pack_padded_sequence(
                x, 
                seq_lengths, 
                batch_first=True, 
                enforce_sorted=False
            )
            
            # LSTM forward pass on packed sequences
            lstm_out_packed, _ = self.lstm(x_packed)
            
            # Unpack sequences
            lstm_out, _ = pad_packed_sequence(lstm_out_packed, batch_first=True)
        else:
            # No masking - process full sequences
            lstm_out, _ = self.lstm(x)
        
        # Apply dropout and output layer
        lstm_out = self.dropout(lstm_out)
        out = self.output_layer(lstm_out).squeeze(-1)  # (batch_size, seq_len)
        
        return out

In [0]:
# ----------------------
# Positional Encoding
# ----------------------
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # [1, max_len, d_model]
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return x

# ----------------------
# StintTransformer
# ----------------------
class StintTransformer(nn.Module):
    def __init__(self, n_drivers, n_teams, n_tyres, n_modes=3,
                 driver_emb_dim=8, team_emb_dim=8, tyre_emb_dim=4, mode_emb_dim=2,
                 n_cont_features=10, d_model=64, nhead=4, num_layers=2,
                 dim_feedforward=128, dropout=0.3):
        super().__init__()
        self.driver_emb = nn.Embedding(n_drivers, driver_emb_dim)
        self.team_emb = nn.Embedding(n_teams, team_emb_dim)
        self.tyre_emb = nn.Embedding(n_tyres, tyre_emb_dim)
        self.mode_emb = nn.Embedding(n_modes, mode_emb_dim)

        input_dim = n_cont_features + driver_emb_dim + team_emb_dim + tyre_emb_dim + mode_emb_dim
        self.input_proj = nn.Linear(input_dim, d_model)
        self.pos_encoder = PositionalEncoding(d_model)

        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead,
                                                   dim_feedforward=dim_feedforward,
                                                   dropout=dropout,
                                                   batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.output_layer = nn.Linear(d_model, 1)

    def forward(self, cont_feats, driver_idx, team_idx, tyre_idx, mode_idx, mask=None):
        batch_size, seq_len, _ = cont_feats.size()
        driver_emb = self.driver_emb(driver_idx).unsqueeze(1).repeat(1, seq_len, 1)
        team_emb = self.team_emb(team_idx).unsqueeze(1).repeat(1, seq_len, 1)
        tyre_emb = self.tyre_emb(tyre_idx)
        mode_emb = self.mode_emb(mode_idx)
        x = torch.cat([cont_feats, driver_emb, team_emb, tyre_emb, mode_emb], dim=-1)
        x = self.input_proj(x)
        x = self.pos_encoder(x)
        key_padding_mask = mask if mask is not None else None
        x = self.transformer(x, src_key_padding_mask=key_padding_mask)
        out = self.output_layer(x).squeeze(-1)
        return out

In [0]:
class StintGRU(nn.Module):
    """
    GRU variant (similar to LSTM but with simpler gating mechanism).
    Often faster than LSTM with similar performance.
    """
    def __init__(self, n_drivers, n_teams, n_tyres, n_modes=3,
                 driver_emb_dim=8, team_emb_dim=8, tyre_emb_dim=4, mode_emb_dim=2,
                 n_cont_features=10, hidden_dim=64, num_layers=2, dropout=0.3):
        super().__init__()
        
        self.driver_emb = nn.Embedding(n_drivers, driver_emb_dim)
        self.team_emb = nn.Embedding(n_teams, team_emb_dim)
        self.tyre_emb = nn.Embedding(n_tyres, tyre_emb_dim)
        self.mode_emb = nn.Embedding(n_modes, mode_emb_dim)
        
        input_dim = n_cont_features + driver_emb_dim + team_emb_dim + tyre_emb_dim + mode_emb_dim
        
        self.gru = nn.GRU(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=False
        )
        
        self.output_layer = nn.Linear(hidden_dim, 1)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, cont_feats, driver_idx, team_idx, tyre_idx, mode_idx, mask=None):
        batch_size, seq_len, _ = cont_feats.size()
        
        driver_emb = self.driver_emb(driver_idx).unsqueeze(1).repeat(1, seq_len, 1)
        team_emb = self.team_emb(team_idx).unsqueeze(1).repeat(1, seq_len, 1)
        tyre_emb = self.tyre_emb(tyre_idx)
        mode_emb = self.mode_emb(mode_idx)
        
        x = torch.cat([cont_feats, driver_emb, team_emb, tyre_emb, mode_emb], dim=-1)
        
        if mask is not None:
            seq_lengths = (~mask).sum(dim=1).cpu()
            x_packed = pack_padded_sequence(x, seq_lengths, batch_first=True, enforce_sorted=False)
            gru_out_packed, _ = self.gru(x_packed)
            gru_out, _ = pad_packed_sequence(gru_out_packed, batch_first=True)
        else:
            gru_out, _ = self.gru(x)
        
        gru_out = self.dropout(gru_out)
        out = self.output_layer(gru_out).squeeze(-1)
        
        return out